# Session 17 · GLMs in statsmodels

Work top to bottom and run each cell with **Shift+Enter**. Cells under a **Your turn** heading contain `...` placeholders: replace them, run the cell, then run the check beneath it. Checks print `PASS` or `NOT YET` and never stop the notebook. Solutions are in the matching `_solutions` notebook.

The data is simulated inside the notebook from a **known** model, so you can see whether each method recovers the truth.

scikit-learn is built for prediction. **statsmodels** is built for inference: standard errors, confidence intervals and p-values, and it understands R-style formulas. It is the Python tool closest to `glm()`.

In [14]:
def check(label, get, expected=True):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [15]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(16)
n = 5_000
members = pd.DataFrame({
    "age": rng.integers(18, 80, n),
    "tenure_years": rng.uniform(0, 10, n).round(1),
    "tier": rng.choice(["basic", "mid", "top"], n, p=[0.5, 0.35, 0.15]),
    "channel": rng.choice(["direct", "broker", "online"], n, p=[0.5, 0.2, 0.3]),
})

# The truth: log-odds of churning in the year
true_logit = (
    -0.8
    - 0.02 * (members["age"] - 45)
    - 0.15 * members["tenure_years"]
    + members["tier"].map({"basic": 0, "mid": -0.3, "top": -0.7})
    + members["channel"].map({"direct": 0, "broker": 0.3, "online": 0.5})
)
members["churned"] = (rng.random(n) < 1 / (1 + np.exp(-true_logit))).astype(int)

# A second outcome for Poisson models: claims in the year, given exposure
members["exposure"] = rng.uniform(0.2, 1.0, n).round(2)
true_log_rate = (np.log(0.8) + 0.015 * (members["age"] - 45)
                 + members["tier"].map({"basic": 0, "mid": 0.2, "top": 0.4}))
members["claims"] = rng.poisson(members["exposure"] * np.exp(true_log_rate))

print(f"{members['churned'].mean():.1%} churned")
members.head()

18.9% churned


,age,tenure_years,tier,channel,churned,exposure,claims
0,51,7.0,top,broker,0,0.38,1
1,53,3.0,basic,online,0,0.45,0
2,69,4.6,basic,online,0,0.67,2
3,44,0.8,mid,broker,0,0.87,4
4,65,7.8,basic,direct,0,0.88,3


In [16]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

## 1. A binomial GLM with a formula

```r
glm(churned ~ age + tenure_years + tier + channel, family = binomial, data = members)
```

becomes

```python
smf.glm("churned ~ age + tenure_years + C(tier) + C(channel, Treatment('direct'))",
        data=members, family=sm.families.Binomial()).fit()
```

`C()` marks a categorical column. Its default reference level is the first alphabetically, so `Treatment('direct')` sets the reference explicitly, like `relevel()` in R.

### Your turn 1.1

Fit `glm_bin` and print its summary.

In [17]:
glm_bin = smf.glm(
    "churned ~ age + tenure_years + C(tier) + C(channel, Treatment('direct'))",
    data=members, family=sm.families.Binomial()).fit()

print(glm_bin.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                churned   No. Observations:                 5000
Model:                            GLM   Df Residuals:                     4993
Model Family:                Binomial   Df Model:                            6
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2263.2
Date:                Sat, 21 Nov 2026   Deviance:                       4526.4
Time:                        14:39:40   Pearson chi2:                 5.02e+03
No. Iterations:                     5   Pseudo R-squ. (CS):            0.06114
Covariance Type:            nonrobust                                         
                                                coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------

In [18]:
check("1.1 tier top near -0.7", lambda: abs(glm_bin.params["C(tier)[T.top]"] - (-0.7)) < 0.3)
check("1.1 age near -0.02", lambda: abs(glm_bin.params["age"] - (-0.02)) < 0.01)
check("1.1 online vs direct near 0.5", lambda: abs(glm_bin.params["C(channel, Treatment('direct'))[T.online]"] - 0.5) < 0.3)

PASS     1.1 tier top near -0.7
PASS     1.1 age near -0.02
PASS     1.1 online vs direct near 0.5


## 2. Odds ratios with confidence intervals

`exp()` of a log-odds coefficient is an odds ratio. `conf_int()` returns the lower and upper bounds as two columns.

### Your turn 2.1

Build `or_table` with columns `odds_ratio`, `lower` and `upper`: `np.exp()` of the parameters and of the confidence interval bounds.

In [19]:
or_table = pd.DataFrame({
    'odds_ratio': np.exp(glm_bin.params),
    'lower': np.exp(glm_bin.conf_int().iloc[:, 0]),
    'upper': np.exp(glm_bin.conf_int().iloc[:, 1])
})
print(or_table)

                                           odds_ratio     lower     upper
Intercept                                    1.107143  0.855867  1.432191
C(tier)[T.mid]                               0.780394  0.664721  0.916196
C(tier)[T.top]                               0.561993  0.442959  0.713015
C(channel, Treatment('direct'))[T.broker]    1.613495  1.331879  1.954657
C(channel, Treatment('direct'))[T.online]    1.823355  1.542286  2.155646
age                                          0.978707  0.974617  0.982814
tenure_years                                 0.858254  0.835906  0.881201


In [20]:
check("2.1 top tier lowers churn odds", lambda: or_table.loc["C(tier)[T.top]", "upper"] < 1)

PASS     2.1 top tier lowers churn odds


## 3. A Poisson GLM with an exposure offset

Claim counts depend on how long each member was insured. In R you would write `glm(claims ~ age + tier, family = poisson, offset = log(exposure))`. statsmodels takes the offset as an argument.

### Your turn 3.1

Fit `glm_pois`: `claims ~ age + C(tier)`, Poisson family, `offset=np.log(members["exposure"])`.

In [21]:
glm_pois = smf.glm(
    "claims ~ age + C(tier)",
    data=members, family=sm.families.Poisson(), 
    offset=np.log(members["exposure"])).fit()

print(glm_pois.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 claims   No. Observations:                 5000
Model:                            GLM   Df Residuals:                     4996
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4938.8
Date:                Sat, 21 Nov 2026   Deviance:                       5071.5
Time:                        14:39:40   Pearson chi2:                 5.10e+03
No. Iterations:                     5   Pseudo R-squ. (CS):            0.05239
Covariance Type:            nonrobust                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -0.8607      0.062    -13.

In [22]:
check("3.1 age near 0.015", lambda: abs(glm_pois.params["age"] - 0.015) < 0.005)
check("3.1 tier top near 0.4", lambda: abs(glm_pois.params["C(tier)[T.top]"] - 0.4) < 0.15)

PASS     3.1 age near 0.015
PASS     3.1 tier top near 0.4


## 4. Check against R

Save the data, run the R script `glm_check.R` (copy it from the Session 17 folder into `sessions/`, next to this notebook), and compare. The same model on the same data should agree to many decimal places.

One catch: statsmodels lists categorical terms **before** numeric ones, while R keeps your formula order. Comparing by position would line up the wrong coefficients, so the cell below matches them **by name**, a habit worth keeping.

In the VS Code terminal: `cd sessions && Rscript glm_check.R && cd ..`

In [23]:
members.to_csv("glm_data.csv", index=False)
print("Saved glm_data.csv. Now run: cd sessions && Rscript glm_check.R && cd ..")

Saved glm_data.csv. Now run: cd sessions && Rscript glm_check.R && cd ..


In [24]:
import os

if os.path.exists("r_glm_binomial.csv"):
    to_statsmodels = {
        "(Intercept)": "Intercept", "age": "age", "tenure_years": "tenure_years",
        "tiermid": "C(tier)[T.mid]", "tiertop": "C(tier)[T.top]",
        "channelbroker": "C(channel, Treatment('direct'))[T.broker]",
        "channelonline": "C(channel, Treatment('direct'))[T.online]",
    }
    r_bin = pd.read_csv("r_glm_binomial.csv").assign(term=lambda d: d["term"].map(to_statsmodels)).set_index("term")["estimate"]
    r_pois = pd.read_csv("r_glm_poisson.csv").assign(term=lambda d: d["term"].map(to_statsmodels)).set_index("term")["estimate"]
    if hasattr(glm_bin, "params"):
        print(pd.DataFrame({"python": glm_bin.params, "r": r_bin}).round(6))
    check("4 binomial matches R", lambda: np.allclose(glm_bin.params, r_bin.reindex(glm_bin.params.index), atol=1e-6))
    check("4 Poisson matches R", lambda: np.allclose(glm_pois.params, r_pois.reindex(glm_pois.params.index), atol=1e-6))
else:
    print("Run glm_check.R first, then rerun this cell.")

                                             python         r
C(channel, Treatment('direct'))[T.broker]  0.478403  0.478403
C(channel, Treatment('direct'))[T.online]  0.600678  0.600678
C(tier)[T.mid]                            -0.247956 -0.247956
C(tier)[T.top]                            -0.576266 -0.576266
Intercept                                  0.101782  0.101782
age                                       -0.021523 -0.021523
tenure_years                              -0.152855 -0.152855
PASS     4 binomial matches R
PASS     4 Poisson matches R


## 5. A preview of Milestone 3

Changing the link function is one argument. A **complementary log-log** link, `sm.families.Binomial(link=sm.families.links.CLogLog())`, turns a binomial GLM on monthly rows into a discrete-time proportional hazards model. That is exactly the model your Milestone 3 simulator is built on.

## Wrap-up

Commit from VS Code (message: `Session 17: GLMs in statsmodels`). Add `glm_data.csv` and `r_glm_*.csv` to `.gitignore`, since they are generated.